# 각 파일에서 6가지 점수 데이터 불러오기

In [3]:
# -*- coding: utf-8 -*-
"""
EWS 종합 위험 스코어용 변수 통합 스크립트 (수정본)
==========================================

[수정 내역]
 1. minmax_by_year 중복곱셈(*100*100 -> 0~10000 스케일) 버그 수정
    -> minmax() 내부에서 이미 *100을 하므로, minmax_by_year에서는 추가로 곱하지 않음
 2. 최종 출력 시 모든 점수 컬럼을 소수점 둘째자리까지 반올림
 3. (참고) 회사명 결측 / 기준연도-회계년도 불일치는 코드 버그가 아니라
    데이터셋 간 표본범위 차이 및 재무자료 입수 시차(reporting lag) 때문입니다.
    -> 자세한 설명은 채팅 답변 참고
"""

import pandas as pd
import numpy as np
import os

# ------------------------------------------------------------------
# 0. 설정
# ------------------------------------------------------------------

OUT_PATH = os.path.join("22번 스코어 산출\통합_스코어_데이터.csv")

# True로 바꾸면 기업단위 변수(기업베타/부실확률/diff)도
# '연도별 횡단면 정규화' 대신 '10개 연도 전체 통합 min-max'를 사용합니다.
GLOBAL_MINMAX = False


def read_csv_safe(filename, **kwargs):
    """사업자등록번호를 문자열(10자리 zfill)로 안전하게 읽는 CSV 로더"""
    path = os.path.join(filename)
    df = pd.read_csv(path, dtype={"사업자등록번호": str}, **kwargs)
    if "사업자등록번호" in df.columns:
        df["사업자등록번호"] = df["사업자등록번호"].str.zfill(10)
    return df


def minmax(s: pd.Series) -> pd.Series:
    """NaN을 무시하는 0~100 스케일 min-max. max==min이면 50으로 처리(분산 0)."""
    mn, mx = s.min(skipna=True), s.max(skipna=True)
    if pd.isna(mn) or pd.isna(mx) or mx == mn:
        return pd.Series(np.where(s.notna(), 50.0, np.nan), index=s.index)
    return (s - mn) / (mx - mn) * 100


def minmax_by_year(df, value_col, year_col="연도"):
    """연도별(횡단면) 0~100 스케일 min-max.
    주의: minmax() 내부에서 이미 *100을 하므로 여기서 추가로 곱하면 안 됨 (중복곱셈 버그 수정됨)
    """
    return df.groupby(year_col)[value_col].transform(minmax)


# ------------------------------------------------------------------
# 1. 베이스 테이블: lifecycle_scored_yearly_minmax.csv
#    (사업자등록번호, 기준연도) 단위가 고유(unique) -> '기준연도'를 '연도'로 사용
# ------------------------------------------------------------------
lifecycle = read_csv_safe("20번. 기업 생애주기\lifecycle_scored_yearly_minmax.csv")

base = lifecycle.rename(columns={"기준연도": "연도"})[
    ["사업자등록번호", "연도", "회계년도", "생애주기_최종", "생애주기_점수", "부실라벨_ICR3년"]
].copy()

# ------------------------------------------------------------------
# 2. 마이클 포터 5F (산업 단위, 도소매업) -> 연도 기준 broadcast
# ------------------------------------------------------------------
porter = read_csv_safe("19번 마이클 포터\마이클 포터 5F_도소매업.csv")
porter = porter[["연도", "최종점수"]].copy()
porter["porter_5F_minmax"] = minmax(porter["최종점수"])   # 10개 연도 시계열 min-max
porter = porter[["연도", "porter_5F_minmax"]]

# ------------------------------------------------------------------
# 3. 산업충격민감도_OLS (산업 단위, 도소매업) -> 연도 기준 broadcast
# ------------------------------------------------------------------
ind_ols = read_csv_safe("18번 산업별 충격민감도\산업충격민감도_OLS.csv")
ind_ols = ind_ols.rename(columns={"테스트_연도": "연도"})[["연도", "beta_i"]].copy()
ind_ols["산업베타_minmax"] = minmax(ind_ols["beta_i"])    # 10개 연도 시계열 min-max
ind_ols = ind_ols[["연도", "산업베타_minmax"]]

# ------------------------------------------------------------------
# 4. 충격민감도_OLS (기업 단위) -> (사업자등록번호, 연도) 기준
# ------------------------------------------------------------------
firm_ols = read_csv_safe("17번 기업별 충격민감도\충격민감도_OLS.csv")
firm_ols = firm_ols.rename(columns={"테스트_연도": "연도"})[
    ["사업자등록번호", "회사명", "연도", "beta_i"]
].copy()

if GLOBAL_MINMAX:
    firm_ols["기업베타_minmax"] = minmax(firm_ols["beta_i"])
else:
    firm_ols["기업베타_minmax"] = minmax_by_year(firm_ols, "beta_i")

firm_ols_for_merge = firm_ols[["사업자등록번호", "연도", "기업베타_minmax"]]
firm_name_map = firm_ols[["사업자등록번호", "회사명"]].drop_duplicates(subset=["사업자등록번호"])

# ------------------------------------------------------------------
# 5. 부실확률 차이값 (기업 단위, wide -> long 변환)
#    연도 범위: 2015~2024 (diff_2014_2015 ~ diff_2023_2024 이용)
# ------------------------------------------------------------------
prob = read_csv_safe(r"21번. 기업 PD 변화율\2015-2024_기업_부실확률_차이값.csv")

years = range(2015, 2025)
long_rows = []
for y in years:
    prob_col = f"prob_{y}"
    diff_col = f"diff_{y-1}_{y}"
    tmp = prob[["사업자등록번호", "회사명", prob_col, diff_col]].copy()
    tmp.columns = ["사업자등록번호", "회사명", "prob", "diff"]
    tmp["연도"] = y
    long_rows.append(tmp)

prob_long = pd.concat(long_rows, ignore_index=True)

if GLOBAL_MINMAX:
    prob_long["부실확률_minmax"] = minmax(prob_long["prob"])
    prob_long["부실확률변화_minmax"] = minmax(prob_long["diff"])
else:
    prob_long["부실확률_minmax"] = minmax_by_year(prob_long, "prob")
    prob_long["부실확률변화_minmax"] = minmax_by_year(prob_long, "diff")

prob_for_merge = prob_long[
    ["사업자등록번호", "회사명", "연도", "부실확률_minmax", "부실확률변화_minmax"]
]

# ------------------------------------------------------------------
# 6. 전체 병합
# ------------------------------------------------------------------
df = base.copy()

# 6-1. 산업 단위 변수 (연도 기준 broadcast)
df = df.merge(porter, on="연도", how="left")
df = df.merge(ind_ols, on="연도", how="left")

# 6-2. 기업 단위 변수
df = df.merge(firm_ols_for_merge, on=["사업자등록번호", "연도"], how="left")
df = df.merge(prob_for_merge, on=["사업자등록번호", "연도"], how="left", suffixes=("", "_prob"))

# 회사명 채우기: 부실확률 파일의 회사명을 우선 사용하고, 없는 경우만 충격민감도_OLS 매핑으로 보강
# (※ if/else로 분기하면, df에 '회사명'이 아직 없을 때 merge 결과 컬럼명이 '회사명_prob'가 아닌
#    '회사명'으로 들어가 else 분기가 타면서 기존 값을 덮어써버리는 버그가 있어 -> 항상 combine_first로 통일)
df["회사명"] = df["회사명"].combine_first(
    df["사업자등록번호"].map(firm_name_map.set_index("사업자등록번호")["회사명"])
)

# ------------------------------------------------------------------
# 6-3. 최종 출력용 컬럼명으로 변경
# ------------------------------------------------------------------
RENAME_MAP = {
    "porter_5F_minmax": "Porter5F",
    "생애주기_점수": "생애주기점수",
    "산업베타_minmax": "산업충격민감도",
    "기업베타_minmax": "기업충격민감도",
    "부실확률_minmax": "부실확률",
    "부실확률변화_minmax": "부실확률변화",
}
df = df.rename(columns=RENAME_MAP)

# ------------------------------------------------------------------
# 7. 최종 컬럼 정리, 소수점 둘째자리 반올림 및 저장
# ------------------------------------------------------------------
final_cols = [
    "사업자등록번호", "회사명", "연도", 
    "생애주기_최종", "부실라벨_ICR3년",
    "Porter5F",          
    "생애주기점수",        
    "산업충격민감도",      
    "기업충격민감도",     
    "부실확률",            
    "부실확률변화",       
]

df_final = df[final_cols].sort_values(["사업자등록번호", "연도"]).reset_index(drop=True)

# 점수 컬럼만 소수점 둘째자리로 반올림
score_cols = [
    "생애주기점수", "Porter5F", "산업충격민감도",
    "기업충격민감도", "부실확률", "부실확률변화",
]
df_final[score_cols] = df_final[score_cols].round(2)

# ------------------------------------------------------------------
# 7-1. '부실확률'이 없는 행 삭제
# ------------------------------------------------------------------
before_n = len(df_final)
df_final = df_final.dropna(subset=["부실확률"]).reset_index(drop=True)
after_n = len(df_final)
print(f"\n'부실확률' 결측 행 삭제: {before_n} -> {after_n} ({before_n - after_n}행 제거)")

df_final.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")

print(f"완료: {OUT_PATH}")
print(f"행/열: {df_final.shape}")
print(df_final.head(10))
print("\n[결측치 비율]")
print(df_final.isna().mean().round(3))
print("\n[점수 컬럼 범위 확인 - 정상적으로 0~100 사이여야 함]")
print(df_final[score_cols].describe().loc[['min','max']])

<>:58: SyntaxWarning: invalid escape sequence '\l'
<>:58: SyntaxWarning: invalid escape sequence '\l'
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_5260\2687088149.py:58: SyntaxWarning: invalid escape sequence '\l'
  lifecycle = read_csv_safe("20번. 기업 생애주기\lifecycle_scored_yearly_minmax.csv")



'부실확률' 결측 행 삭제: 48057 -> 32801 (15256행 제거)
완료: 22번 스코어 산출\통합_스코어_데이터.csv
행/열: (32801, 11)
      사업자등록번호          회사명    연도 생애주기_최종  부실라벨_ICR3년  Porter5F  생애주기점수  \
0  0008110041  엘브이엠씨홀딩스(주)  2015     도입기           0    100.00    75.7   
1  0008110041  엘브이엠씨홀딩스(주)  2016     도입기           0     56.26    80.6   
2  0008110041  엘브이엠씨홀딩스(주)  2017     성장기           0     34.84     1.1   
3  0008110041  엘브이엠씨홀딩스(주)  2018     성장기           0     25.54     4.0   
4  0008110041  엘브이엠씨홀딩스(주)  2019     성장기           0     56.69     0.2   
5  0008110041  엘브이엠씨홀딩스(주)  2020     성장기           1      0.00     0.0   
6  1018106586      서원물산(주)  2015     도입기           1    100.00    75.7   
7  1018116269   현대코퍼레이션(주)  2015     성숙기           0    100.00     3.2   
8  1018116269   현대코퍼레이션(주)  2016     성장기           0     56.26     0.3   
9  1018116269   현대코퍼레이션(주)  2017     조정기           0     34.84    21.8   

   산업충격민감도  기업충격민감도   부실확률  부실확률변화  
0     0.00    44.36   0.19   49.99  
1    42.55    48.52 

# 신용등급 존재하는 데이터 merge, 그리드 서치했을때의 최적 가중치

In [ ]:
# -*- coding: utf-8 -*-
"""
신용등급 병합 + 최적 가중치 그리드서치
========================================
[흐름]
 1. 통합_스코어_데이터.csv 로드
 2. 신용등급 파일 전처리 (NICE, 장기등급, 연말등급)
 3. score 연도 + 1 = 신용등급 연도 로 병합 (2024 점수 ↔ 2025 신용등급)
 4. 신용등급 정제 → 등급_순위 변환
 5. 그리드서치: Spearman 상관으로 최적 가중치 탐색
"""

import re
import pandas as pd
import numpy as np
from scipy.stats import spearmanr

# ============================================================
# 1. 통합 스코어 데이터 로드
# ============================================================
score_df = pd.read_csv("22번 스코어 산출/통합_스코어_데이터.csv",
                       dtype={"사업자등록번호": str})
score_df["사업자등록번호"] = score_df["사업자등록번호"].str.zfill(10)

print(f"score_df shape: {score_df.shape}")
print(f"score_df 컬럼: {score_df.columns.tolist()}")

# ============================================================
# 2. 신용등급 파일 전처리
# ============================================================
credit_df = pd.read_excel(r"..\데이터수집\신용등급\상장사 신용등급.xlsx",
                          dtype={"사업자등록번호": str})

credit_df["사업자등록번호"] = (
    credit_df["사업자등록번호"]
    .str.strip()
    .str.replace("-", "", regex=False)
    .str.zfill(10)
)

# 회계년도 파싱: "2015/12" → 연도=2015, 월=12
credit_df["연도"] = credit_df["회계년도"].astype(str).str.split("/").str[0].astype(int)
credit_df["월"]   = credit_df["회계년도"].astype(str).str.split("/").str[1].astype(int)

# NICE신용평가(평가사구분=10)만 사용
credit_filtered = credit_df[credit_df["평가사구분"].isin([10])].copy()

# ------------------------------------------------------------
# 장기등급만 필터링 (단기 CP등급 제외: 숫자 포함된 등급)
# ------------------------------------------------------------
def is_long_term(rating):
    if pd.isna(rating):
        return False
    grade = str(rating).split("/")[0]
    return not re.search(r"\d", grade)

credit_filtered = credit_filtered[
    credit_filtered["신용등급"].apply(is_long_term)
].copy()

print(f"\n장기등급 필터링 후 unique 등급값:")
print(sorted(credit_filtered["신용등급"].dropna().unique()))

# ------------------------------------------------------------
# 연중 변경 시 연말(가장 최근 월) 등급 채택
# ------------------------------------------------------------
idx_latest = (
    credit_filtered
    .groupby(["사업자등록번호", "연도"])["월"]
    .idxmax()
)
credit_filtered = credit_filtered.loc[idx_latest].copy()

# ------------------------------------------------------------
# 동일 (사업자등록번호, 연도) 내 등급 불일치 처리
# ------------------------------------------------------------
credit_filtered["등급_base"] = credit_filtered["신용등급"].str.split("/").str[0]
credit_filtered = credit_filtered.drop_duplicates(
    subset=["사업자등록번호", "연도", "월", "평가사명 및 등급", "신용등급"]
)

def collapse_if_same(group):
    unique_base = group["등급_base"].dropna().unique()
    if len(unique_base) <= 1:
        return group.iloc[[0]]
    else:
        biz_no = group["사업자등록번호"].iloc[0]
        year   = group["연도"].iloc[0]
        print(f"\n[신용등급 불일치] 사업자등록번호: {biz_no}, 연도: {year}")
        print(group[["월", "신용등급", "평가사명 및 등급"]].to_string(index=False))
        return group

credit_filtered = (
    credit_filtered
    .groupby(["사업자등록번호", "연도"], group_keys=False)
    .apply(collapse_if_same)
    .reset_index(drop=True)
)

# ✅ credit 연도를 미리 rename → 병합 시 컬럼 충돌 방지
credit_for_merge = credit_filtered[
    ["사업자등록번호", "연도", "평가사명 및 등급", "신용등급", "평가사구분"]
].copy()
credit_for_merge = credit_for_merge.rename(columns={"연도": "연도_신용등급"})

# ============================================================
# 3. 병합: score 연도 + 1 = 신용등급 연도
#    (2024년 재무 기반 점수 ↔ 2025년 신용등급 비교)
# ============================================================
score_df["연도_신용등급"] = score_df["연도"] + 1

result_df = score_df.merge(
    credit_for_merge,
    on=["사업자등록번호", "연도_신용등급"],
    how="inner"
)

print(f"\n병합 후 컬럼: {result_df.columns.tolist()}")
print(f"병합 후 shape: {result_df.shape}")

# ============================================================
# 4. 신용등급 정제 → 등급_순위 변환
# ============================================================
merged = result_df.copy()

# 4-1. 전망 텍스트 제거 후 대문자 통일
merged["등급_정제"] = (
    merged["신용등급"]
    .str.split(r"[/(\s]").str[0]
    .str.strip()
    .str.upper()
)

# 4-2. 구분자 없이 붙어있는 케이스 수동 처리 (정제 후 replace 순서 중요!)
manual_fix = {
    "AA+STABLE":  "AA+",
    "AA+POSTIVE": "AA+",
    "AASTABLE":   "AA",
    "AA-STABLE":  "AA-",
    "A+STABLE":   "A+",
}
merged["등급_정제"] = merged["등급_정제"].replace(manual_fix)

# 4-3. 등급 → 숫자 순위 매핑 (AAA=1 최우량, D=22 최하)
grade_order = [
    "AAA",
    "AA+", "AA", "AA-",
    "A+",  "A",  "A-",
    "BBB+","BBB","BBB-",
    "BB+", "BB", "BB-",
    "B+",  "B",  "B-",
    "CCC+","CCC","CCC-",
    "CC",  "C",  "D"
]
grade_map = {g: i+1 for i, g in enumerate(grade_order)}
merged["등급_순위"] = merged["등급_정제"].map(grade_map)

# 4-4. 매핑 안 된 등급 확인
unmapped = merged[merged["등급_순위"].isna()]["등급_정제"].unique()
if len(unmapped) > 0:
    print(f"\n[경고] 매핑 안 된 등급: {unmapped}")
    print("→ manual_fix 딕셔너리에 추가 필요")
else:
    print("\n모든 등급 매핑 완료 ✅")

merged = merged.dropna(subset=["등급_순위"]).copy()
merged.loc[:, "등급_순위"] = merged["등급_순위"].astype(int)

print(f"최종 merged shape: {merged.shape}")
print(merged[["회사명", "신용등급", "등급_정제", "등급_순위"]].head(10))

# ============================================================
# 5. 그리드서치 전 컬럼 존재 확인
# ============================================================
pd_col    = "부실확률"
other_cols = ["Porter5F", "생애주기점수", "산업충격민감도", "기업충격민감도", "부실확률변화"]
need_cols  = [pd_col] + other_cols

missing = [c for c in need_cols if c not in merged.columns]
if missing:
    print(f"\n[경고] merged에 없는 컬럼: {missing}")
    raise ValueError("위 컬럼을 확인하세요. 스코어 파일에 해당 컬럼이 있어야 합니다.")
else:
    print(f"\n그리드서치 필요 컬럼 모두 확인 완료 ✅")
    print(f"샘플 수: {len(merged)}개")

# ============================================================
# 6. 가중치 조합 생성 함수
#    - 0.05 단위, 모든 변수 최소 5% 보장
#    - 부실확률(pd_col): 50%~95% (10~19단위)
#    - 나머지 5개 변수: 각 최소 5%
# ============================================================
def gen_combos(total_units, n_vars):
    """total_units개를 n_vars개 변수에 비음수로 분배하는 모든 조합"""
    if n_vars == 1:
        yield (total_units,)
        return
    for i in range(total_units + 1):
        for rest in gen_combos(total_units - i, n_vars - 1):
            yield (i,) + rest

# ============================================================
# 7. 그리드서치: 모든 가중치 조합에 대해 Spearman 상관 계산
# ============================================================
y       = merged["등급_순위"].values
n_other = len(other_cols)
results = []

# pd_units: 10(=0.50) ~ 14(=0.70) → 나머지 5개가 각 최소 1단위(0.05) 확보
for pd_units in range(10, 15):
    w_pd = round(pd_units * 0.05, 2)

    remaining_total = 20 - pd_units      # 나머지 5개 변수에 배분할 총 단위
    extra_units     = remaining_total - n_other  # 각 1단위 기본 배정 후 남는 단위

    if extra_units < 0:
        continue

    for extra_combo in gen_combos(extra_units, n_other):
        weights_other = [round((1 + e) * 0.05, 2) for e in extra_combo]

        # 가중치 합 검증 (부동소수점 오차 허용)
        total_w = round(w_pd + sum(weights_other), 2)
        if abs(total_w - 1.0) > 0.01:
            continue

        score = merged[pd_col].values * w_pd
        for col, w in zip(other_cols, weights_other):
            score = score + merged[col].values * w

        rho, p = spearmanr(score, y)

        results.append({
            "부실확률_w":       w_pd,
            **{f"{c}_w": w for c, w in zip(other_cols, weights_other)},
            "가중치합":         total_w,
            "spearman_rho":    round(rho, 4),
            "p_value":         round(p, 4),
        })

res_df = pd.DataFrame(results)

# ============================================================
# 8. 결과 정렬 및 출력
# ============================================================
# p_value 기준 정렬 (유의한 것 우선), 동순위는 |rho| 내림차순
res_df["abs_rho"] = res_df["spearman_rho"].abs()
res_df_sorted = res_df.sort_values(["p_value", "abs_rho"], ascending=[True, False])

print(f"\n총 탐색 조합 수: {len(res_df)}")
print(f"5% 기준 우연히 p<0.05가 나올 것으로 예상되는 조합 수: 약 {int(len(res_df)*0.05)}개")
print(f"실제 p<0.05인 조합 수: {(res_df['p_value'] < 0.05).sum()}개")

print("\n[최적 조합 상위 10개 (p-value 기준)]")
display_cols = ["부실확률_w"] + [f"{c}_w" for c in other_cols] + ["가중치합", "spearman_rho", "p_value"]
print(res_df_sorted[display_cols].head(10).to_string(index=False))

# 최적 가중치 1개 출력
best = res_df_sorted.iloc[0]
print("\n" + "="*50)
print("★ 최적 가중치 조합 ★")
print("="*50)
print(f"  부실확률        : {best['부실확률_w']:.2f}")
for c in other_cols:
    print(f"  {c:<12}: {best[f'{c}_w']:.2f}")
print(f"  가중치 합       : {best['가중치합']:.2f}")
print(f"  Spearman rho    : {best['spearman_rho']:.4f}")
print(f"  p-value         : {best['p_value']:.4f}")

# ============================================================
# 9. 저장
# ============================================================

# ✅ 추가: 신용등급 병합 결과 저장
merged.to_csv(
    "22번 스코어 산출/통합_스코어_신용등급_병합.csv",
    index=False, encoding="utf-8-sig"
)
print("신용등급 병합 결과 저장 완료: 22번 스코어 산출/통합_스코어_신용등급_병합.csv")

# 기존: 그리드서치 결과 저장
res_df_sorted[display_cols].to_csv(
    "22번 스코어 산출/가중치_그리드서치_결과.csv",
    index=False, encoding="utf-8-sig"
)
print("그리드서치 결과 저장 완료: 22번 스코어 산출/가중치_그리드서치_결과.csv")

score_df shape: (32801, 11)
score_df 컬럼: ['사업자등록번호', '회사명', '연도', '생애주기_최종', '부실라벨_ICR3년', 'Porter5F', '생애주기점수', '산업충격민감도', '기업충격민감도', '부실확률', '부실확률변화']

장기등급 필터링 후 unique 등급값:
['-', 'A', 'A ', 'A (긍정적)', 'A (부정적)', 'A (안정적)', 'A / STABLE', 'A(Positive)', 'A(STABLE)', 'A(긍정적)', 'A(부정적)', 'A(안정적)', 'A(하향검토)', 'A+', 'A+ (긍정적)', 'A+ (부정적)', 'A+ (안정적)', 'A+ (하향검토)', 'A+ Positive', 'A+ Stable', 'A+(POSITIVE)', 'A+(Positive)', 'A+(STABLE)', 'A+(부정적)', 'A+(안정적)', 'A+(하향검토)', 'A+/ 부정적', 'A+/ 안정적', 'A+/STABLE', 'A+/긍정적', 'A+/안정적', 'A+STABLE', 'A+↓', 'A-', 'A-  (긍정적)', 'A-  (안정적)', 'A- (긍정적)', 'A- (부정적)', 'A- (안정적)', 'A- / POSITIVE', 'A- / Positive', 'A- STABLE', 'A- sTABLE', 'A-(Positive)', 'A-(STABLE)', 'A-(Stable)', 'A-(긍정적)', 'A-(부정적)', 'A-(안정적)', 'A-(하향검토)', 'A-/Positive', 'A-/STABLE', 'A-/부정적', 'A-/안정적', 'A-↓', 'A/ 안정적', 'A/Positive', 'A/Postive', 'A/STABLE', 'A/부정적', 'A/안정적', 'AA', 'AA (긍정적)', 'AA (부정적)', 'AA (안정적)', 'AA 안정적', 'AA(STABLE)', 'AA(긍정적)', 'AA(부정적)', 'AA(안정적)', 'AA+', 'AA+ (긍정

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_5260\1703558697.py:96: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(collapse_if_same)



병합 후 컬럼: ['사업자등록번호', '회사명', '연도', '생애주기_최종', '부실라벨_ICR3년', 'Porter5F', '생애주기점수', '산업충격민감도', '기업충격민감도', '부실확률', '부실확률변화', '연도_신용등급', '평가사명 및 등급', '신용등급', '평가사구분']
병합 후 shape: (168, 15)

모든 등급 매핑 완료 ✅
최종 merged shape: (168, 17)
          회사명         신용등급 등급_정제  등급_순위
0  현대코퍼레이션(주)    A-/STABLE    A-      7
1  현대코퍼레이션(주)    A-/STABLE    A-      7
2  현대코퍼레이션(주)    A-/STABLE    A-      7
3  현대코퍼레이션(주)    A-/STABLE    A-      7
4  현대코퍼레이션(주)  A-/Positive    A-      7
5  현대코퍼레이션(주)     A/STABLE     A      6
6   롯데하이마트(주)          AA-   AA-      4
7   롯데하이마트(주)          AA-   AA-      4
8   롯데하이마트(주)          AA-   AA-      4
9   롯데하이마트(주)          AA-   AA-      4

그리드서치 필요 컬럼 모두 확인 완료 ✅
샘플 수: 168개

총 탐색 조합 수: 251
5% 기준 우연히 p<0.05가 나올 것으로 예상되는 조합 수: 약 12개
실제 p<0.05인 조합 수: 59개

[최적 조합 상위 10개 (p-value 기준)]
 부실확률_w  Porter5F_w  생애주기점수_w  산업충격민감도_w  기업충격민감도_w  부실확률변화_w  가중치합  spearman_rho  p_value
   0.50        0.05      0.30       0.05       0.05      0.05   1.0        0.2088   0.0066
   0.50 

# 최적 가중치 기반 Score 산출

In [7]:
import pandas as pd

weights_df = pd.read_csv(r"22번 스코어 산출\가중치_그리드서치_결과.csv")
first_row_weights = weights_df.iloc[0]

w_bankruptcy = first_row_weights["부실확률_w"]
w_porter = first_row_weights["Porter5F_w"]
w_lifecycle = first_row_weights["생애주기점수_w"]
w_ind_shock = first_row_weights["산업충격민감도_w"]
w_comp_shock = first_row_weights["기업충격민감도_w"]
w_bankruptcy_chg = first_row_weights["부실확률변화_w"]

score_df = pd.read_csv(r"22번 스코어 산출\통합_스코어_데이터.csv")

score_df["최종합산스코어"] = (
    (score_df["부실확률"] * w_bankruptcy) +
    (score_df["Porter5F"] * w_porter) +
    (score_df["생애주기점수"] * w_lifecycle) +
    (score_df["산업충격민감도"] * w_ind_shock) +
    (score_df["기업충격민감도"] * w_comp_shock) +
    (score_df["부실확률변화"] * w_bankruptcy_chg)
)

score_df["최종합산스코어"] = score_df["최종합산스코어"].round(1)

score_df.to_csv(r"22번 스코어 산출\통합_스코어_데이터.csv", index=False)

In [8]:
import pandas as pd

weights_df = pd.read_csv(r"22번 스코어 산출\가중치_그리드서치_결과.csv")
first_row_weights = weights_df.iloc[0]

w_bankruptcy = first_row_weights["부실확률_w"]
w_porter = first_row_weights["Porter5F_w"]
w_lifecycle = first_row_weights["생애주기점수_w"]
w_ind_shock = first_row_weights["산업충격민감도_w"]
w_comp_shock = first_row_weights["기업충격민감도_w"]
w_bankruptcy_chg = first_row_weights["부실확률변화_w"]

score_df = pd.read_csv(r"22번 스코어 산출\통합_스코어_데이터.csv")

score_df["최종합산스코어"] = (
    (score_df["부실확률"] * w_bankruptcy) +
    (score_df["Porter5F"] * w_porter) +
    (score_df["생애주기점수"] * w_lifecycle) +
    (score_df["산업충격민감도"] * w_ind_shock) +
    (score_df["기업충격민감도"] * w_comp_shock) +
    (score_df["부실확률변화"] * w_bankruptcy_chg)
)

score_df["최종합산스코어"] = score_df["최종합산스코어"].round(1)

merge_target_df = pd.read_csv(r"22번 스코어 산출\통합_스코어_신용등급_병합.csv")

score_subset = score_df[["사업자등록번호", "연도", "최종합산스코어"]]

result_df = pd.merge(
    merge_target_df, 
    score_subset, 
    on=["사업자등록번호", "연도"], 
    how="left"
)

result_df.to_csv(r"22번 스코어 산출\통합_스코어_신용등급_병합.csv", index=False)

# 등급_5가지 부여

In [9]:
import pandas as pd

df = pd.read_csv(r"22번 스코어 산출\통합_스코어_데이터.csv")

def assign_grade_by_score(score):
    if score >= 60:
        return "매우 위험"
    elif score >= 50:
        return "위험"
    elif score >= 40:
        return "중립"
    elif score >= 30:
        return "안정"
    else:
        return "매우 안정"

# 1. 기존 데이터프레임에 신용등급_5단계 칼럼 추가
df["등급_5단계"] = df["최종합산스코어"].apply(assign_grade_by_score)

# 2. 등급별 검증 리포트 출력 (기존 로직 유지)
report = df.groupby("등급_5단계").agg(
    최소점수=("최종합산스코어", "min"),
    최대점수=("최종합산스코어", "max"),
    기업수=("부실라벨_ICR3년", "count"),
    실제부실기업수=("부실라벨_ICR3년", "sum"),
    구간내_실제부실률=("부실라벨_ICR3년", "mean")
).reindex(["매우 위험", "위험", "중립", "안정", "매우 안정"])

print(report)

# 3. 신용등급 칼럼이 추가된 최종 데이터를 파일로 저장
# 원본을 유지하고 싶다면 다른 이름으로 저장하는 것을 권장합니다.
df.to_csv(r"22번 스코어 산출\통합_스코어_데이터.csv", index    =False)

        최소점수  최대점수    기업수  실제부실기업수  구간내_실제부실률
등급_5단계                                       
매우 위험   60.0  96.3   2250      806   0.358222
위험      50.0  59.9    990      193   0.194949
중립      40.0  49.9   3131       39   0.012456
안정      30.0  39.9   4212       21   0.004986
매우 안정    6.5  29.9  22218       31   0.001395


In [10]:
import pandas as pd

# 1. 파일 불러오기
# (이전 단계에서 신용등급_5단계 칼럼을 추가한 파일이거나 원본 파일 모두 사용 가능합니다)
score_df = pd.read_csv(r"22번 스코어 산출\통합_스코어_데이터.csv")
rating_df = pd.read_csv(r"22번 스코어 산출\통합_스코어_신용등급_병합.csv")

# 2. 통합_스코어_신용등급_병합 파일에서 필요한 칼럼만 추출
# 사업자등록번호, 연도와 함께 가져올 '신용등급' 관련 칼럼들을 지정합니다.
# 만약 등급 관련 다른 칼럼도 함께 가져오고 싶다면 리스트에 추가하시면 됩니다.
rating_subset = rating_df[["사업자등록번호", "연도",  "신용등급"]]

# 3. 통합_스코어_데이터(score_df)를 기준으로 왼쪽 결합(Left Merge) 수행
# 이렇게 하면 rating_df에 데이터가 있으면 등급이 들어오고, 없으면 자동으로 NaN 처리가 됩니다.
merged_df = pd.merge(
    score_df, 
    rating_subset, 
    on=["사업자등록번호", "연도"], 
    how="left"
)

# 4. 결과 저장
merged_df.to_csv(r"22번 스코어 산출\통합_스코어_데이터.csv", index=False)